# Joint Service Caching & Task Offloading Simulation in Vehicular Edge Computing


## Subproblems and System


In [1]:
"""
Implements the three-block Alternating Optimization (AO) scheme discussed:

  Subproblem (i)   Task Offloading Decision      -> Branch-and-Bound (BnB)
  Subproblem (ii)  Joint Computing & Bandwidth    -> Interior Point Method (IPM)
  Subproblem (iii) Service Fetching Decision      -> Low-Complexity Service
                                                      Fetching (LCSF)

System:
  - 1 RSU that caches ALL service models, has large compute (F_r_max) and
    shares uplink bandwidth budget B_u among V2I-offloading tasks.
  - Several Service Vehicles (SVs): each caches only a SUBSET of models
    (bounded by storage capacity), has moderate compute (F_v_max), and can
    execute tasks offloaded to it over V2V links.
  - Several Task Vehicles (TVs): generate tasks that require a specific
    service model; each task can be executed LOCALLY (at the TV's own weak
    CPU), at the RSU (V2I), or at a paired Service Vehicle (V2V).
  - If the chosen execution node does not already cache the required model,
    a Service-Fetching decision (LCSF) determines whether the model is
    downloaded in PARALLEL with the task data or SERIALLY after it.

The AO loop iterates:  BnB(y) -> IPM(f,b) -> LCSF(phi) -> (repeat)
until the total delay stabilises or a max-iteration budget is hit.

Author: (simulation code)
"""

"\nImplements the three-block Alternating Optimization (AO) scheme discussed:\n\n  Subproblem (i)   Task Offloading Decision      -> Branch-and-Bound (BnB)\n  Subproblem (ii)  Joint Computing & Bandwidth    -> Interior Point Method (IPM)\n  Subproblem (iii) Service Fetching Decision      -> Low-Complexity Service\n                                                      Fetching (LCSF)\n\nSystem:\n  - 1 RSU that caches ALL service models, has large compute (F_r_max) and\n    shares uplink bandwidth budget B_u among V2I-offloading tasks.\n  - Several Service Vehicles (SVs): each caches only a SUBSET of models\n    (bounded by storage capacity), has moderate compute (F_v_max), and can\n    execute tasks offloaded to it over V2V links.\n  - Several Task Vehicles (TVs): generate tasks that require a specific\n    service model; each task can be executed LOCALLY (at the TV's own weak\n    CPU), at the RSU (V2I), or at a paired Service Vehicle (V2V).\n  - If the chosen execution node does not a

## Packages

In [2]:
import math
import random
import time
import itertools
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
from scipy.optimize import minimize, LinearConstraint, Bounds
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

## 1. Configuration / System Parameters

In [3]:
CFG = dict(
    ROAD_LENGTH_M      = 2000.0,     # 1-D road segment
    RSU_POS_M          = 1000.0,     # RSU fixed at road midpoint
    V2I_RANGE_M        = 450.0,      # RSU coverage radius
    V2V_RANGE_M        = 250.0,      # Vehicle-to-vehicle comm range

    NUM_SERVICE_VEH    = 4,
    NUM_TASK_VEH       = 8,
    NUM_MODELS         = 10,

    MODEL_SIZE_MB      = (50.0, 200.0),     # uniform range
    SV_STORAGE_MB      = 380.0,             # each SV can cache ~2-3 models
    SV_CACHE_COUNT     = 3,                 # each SV pre-caches this many models

    F_RSU_MAX_GHZ      = 24.0,
    F_SV_MAX_GHZ       = 6.0,
    F_LOCAL_GHZ        = 1.2,               # weak on-board CPU of a task vehicle

    BW_TOTAL_MHZ       = 20.0,              # B_u, shared uplink budget (V2I & V2V pools)

    TX_POWER_W         = 0.5,
    NOISE_POWER_W      = 1e-13,
    PATHLOSS_ALPHA     = 3.0,
    REF_DIST_M         = 1.0,
    REF_GAIN           = 1e-3,              # channel gain at reference distance

    TASK_DATA_MB       = (1.0, 6.0),        # input data size (megabits used internally)
    TASK_CYCLES_MC     = (500.0, 2500.0),   # required CPU cycles, in Mega-cycles
    TASK_DEADLINE_S    = (0.4, 2.2),

    NUM_TIME_SLOTS     = 15,
    SLOT_DURATION_S    = 1.0,

    AO_MAX_ITERS       = 6,
    AO_TOL_S           = 1e-3,

    BNB_NODE_CAP       = 15000,             # safety cap -> greedy fallback if exceeded
    EPS                = 1e-6,
)

## 2.  Entities

In [4]:
@dataclass
class Model:
    id: int
    size_mb: float


@dataclass
class Task:
    id: int
    tv_id: int                 # id of the task (generating) vehicle
    model_id: int
    data_mb: float              # input data to transmit if offloaded
    cycles_mc: float            # required CPU cycles (mega-cycles)
    deadline_s: float


@dataclass
class RSU:
    pos: float
    f_max_ghz: float
    cached_models: set


@dataclass
class ServiceVehicle:
    id: int
    pos: float
    vel: float
    f_max_ghz: float
    storage_mb: float
    cached_models: set


@dataclass
class TaskVehicle:
    id: int
    pos: float
    vel: float
    f_local_ghz: float
    cached_models: set = field(default_factory=set)   # usually empty / tiny

## 3. Channel / Mobility Helpers

In [5]:
def path_gain(distance_m: float) -> float:
    """Simple power-law path-loss channel gain model."""
    d = max(distance_m, CFG["REF_DIST_M"])
    return CFG["REF_GAIN"] * (CFG["REF_DIST_M"] / d) ** CFG["PATHLOSS_ALPHA"]


def shannon_rate_mbps(bandwidth_mhz: float, distance_m: float) -> float:
    """Achievable rate (Mbps) over a link of given bandwidth & distance."""
    if bandwidth_mhz <= 0:
        return CFG["EPS"]
    snr = (CFG["TX_POWER_W"] * path_gain(distance_m)) / CFG["NOISE_POWER_W"]
    return bandwidth_mhz * math.log2(1.0 + snr)


def contact_duration_s(pos_a, vel_a, pos_b, vel_b, comm_range) -> float:
    """
    Simplified 1-D contact-window model: time remaining until the two nodes
    (or a node and the fixed RSU) drift outside the communication range.
    """
    dist = abs(pos_a - pos_b)
    if dist > comm_range:
        return 0.0
    rel_speed = abs(vel_a - vel_b)
    if rel_speed < 1e-3:
        return 60.0  # effectively "always connected" for this slot horizon
    remaining_gap = max(comm_range - dist, 0.0)
    return float(np.clip(remaining_gap / rel_speed, 0.0, 60.0))

## 4. Delay Component Functions

In [6]:
def compute_delay_s(cycles_mc: float, freq_ghz: float) -> float:
    freq_ghz = max(freq_ghz, CFG["EPS"])
    return cycles_mc / (freq_ghz * 1000.0)


def trans_delay_s(data_mb: float, rate_mbps: float) -> float:
    rate_mbps = max(rate_mbps, CFG["EPS"])
    return data_mb / rate_mbps


## 5. Scenario Construction

In [7]:
def build_models() -> List[Model]:
    return [Model(i, random.uniform(*CFG["MODEL_SIZE_MB"])) for i in range(CFG["NUM_MODELS"])]


def build_scenario(models: List[Model], num_tv: int = None, num_sv: int = None):
    ntv = num_tv if num_tv is not None else CFG["NUM_TASK_VEH"]
    nsv = num_sv if num_sv is not None else CFG["NUM_SERVICE_VEH"]

    rsu = RSU(pos=CFG["RSU_POS_M"], f_max_ghz=CFG["F_RSU_MAX_GHZ"],
              cached_models=set(m.id for m in models))

    service_vehicles = []
    for i in range(nsv):
        pos = random.uniform(0, CFG["ROAD_LENGTH_M"])
        vel = random.uniform(-20, 20)   # m/s, +/- direction of travel
        cached = set(random.sample(range(CFG["NUM_MODELS"]),
                                    min(CFG["SV_CACHE_COUNT"], CFG["NUM_MODELS"])))
        service_vehicles.append(ServiceVehicle(i, pos, vel, CFG["F_SV_MAX_GHZ"],
                                                 CFG["SV_STORAGE_MB"], cached))

    task_vehicles = []
    for i in range(ntv):
        pos = random.uniform(0, CFG["ROAD_LENGTH_M"])
        vel = random.uniform(-20, 20)
        task_vehicles.append(TaskVehicle(i, pos, vel, CFG["F_LOCAL_GHZ"]))

    return rsu, service_vehicles, task_vehicles


def move_vehicles(service_vehicles, task_vehicles, dt):
    for v in service_vehicles + task_vehicles:
        v.pos += v.vel * dt
        v.pos = float(np.clip(v.pos, 0.0, CFG["ROAD_LENGTH_M"]))


def generate_tasks(task_vehicles, models, slot_idx) -> List[Task]:
    tasks = []
    zipf_weights = np.array([1.0 / (r + 1) for r in range(len(models))])
    zipf_probs = zipf_weights / zipf_weights.sum()
    for tv in task_vehicles:
        model_id = int(np.random.choice(len(models), p=zipf_probs))
        tasks.append(Task(
            id=slot_idx * 1000 + tv.id,
            tv_id=tv.id,
            model_id=model_id,
            data_mb=random.uniform(*CFG["TASK_DATA_MB"]),
            cycles_mc=random.uniform(*CFG["TASK_CYCLES_MC"]),
            deadline_s=random.uniform(*CFG["TASK_DEADLINE_S"]),
        ))
    return tasks

## 6. Candidate Mode Enumeration (feasibility pre-filter -> connectivity C6/C7)

In [8]:
NODE_LOCAL = "local"
NODE_RSU = "rsu"


def get_candidates(task: Task, tv: TaskVehicle, rsu: RSU,
                    service_vehicles: List[ServiceVehicle]) -> List[Tuple[str, float]]:
    candidates = [(NODE_LOCAL, 60.0)]

    tau_v2i = contact_duration_s(tv.pos, tv.vel, rsu.pos, 0.0, CFG["V2I_RANGE_M"])
    if tau_v2i > 0:
        candidates.append((NODE_RSU, tau_v2i))

    for sv in service_vehicles:
        tau_v2v = contact_duration_s(tv.pos, tv.vel, sv.pos, sv.vel, CFG["V2V_RANGE_M"])
        if tau_v2v > 0:
            candidates.append((sv.id, tau_v2v))

    return candidates

## 7. Subproblem (iii): Low-Complexiy Service Fetching (LCSF)

In [9]:
def node_has_model(node_key, model_id, rsu, service_vehicles, task_vehicles) -> bool:
    if node_key == NODE_LOCAL:
        return False
    if node_key == NODE_RSU:
        return model_id in rsu.cached_models
    sv = next(sv for sv in service_vehicles if sv.id == node_key)
    return model_id in sv.cached_models


def lcsf_decide(task: Task, node_key, tv: TaskVehicle, rsu: RSU,
                 service_vehicles: List[ServiceVehicle],
                 models: List[Model], f_alloc_ghz: float, b_offload_mhz: float,
                 b_fetch_mhz: float) -> Dict:
    model = models[task.model_id]

    if node_key == NODE_LOCAL:
        exec_pos, exec_vel = tv.pos, tv.vel
    elif node_key == NODE_RSU:
        exec_pos, exec_vel = rsu.pos, 0.0
    else:
        sv = next(sv for sv in service_vehicles if sv.id == node_key)
        exec_pos, exec_vel = sv.pos, sv.vel

    dist_tv_exec = abs(tv.pos - exec_pos)
    dist_rsu_exec = abs(rsu.pos - exec_pos)

    if node_key == NODE_LOCAL:
        t_off = 0.0
    else:
        rate_off = shannon_rate_mbps(b_offload_mhz, dist_tv_exec)
        t_off = trans_delay_s(task.data_mb, rate_off)

    rate_fetch_from_rsu = shannon_rate_mbps(b_fetch_mhz, dist_rsu_exec)
    t_fetch_parallel = trans_delay_s(model.size_mb, rate_fetch_from_rsu)

    rate_fetch_serial = shannon_rate_mbps(b_fetch_mhz, dist_tv_exec if node_key != NODE_LOCAL else 1.0)
    t_fetch_serial = trans_delay_s(model.size_mb, rate_fetch_serial)

    t_comp = compute_delay_s(task.cycles_mc, f_alloc_ghz)

    t_parallel = max(t_fetch_parallel, t_off) + t_comp
    t_serial = t_off + t_fetch_serial + t_comp

    if t_parallel <= t_serial:
        mode, t_fetch_used, t_total = "parallel", t_fetch_parallel, t_parallel
    else:
        mode, t_fetch_used, t_total = "serial", t_fetch_serial, t_serial

    storage_ok = True
    if node_key not in (NODE_LOCAL,):
        if node_key == NODE_RSU:
            storage_ok = True
        else:
            sv = next(sv for sv in service_vehicles if sv.id == node_key)
            used = sum(models[m].size_mb for m in sv.cached_models)
            storage_ok = (used + model.size_mb) <= sv.storage_mb

    deadline_ok = t_total <= task.deadline_s

    return dict(mode=mode, t_off=t_off, t_fetch=t_fetch_used, t_comp=t_comp,
                t_total=t_total, feasible=(storage_ok and deadline_ok),
                storage_ok=storage_ok, deadline_ok=deadline_ok)

## 8. SUBPROBLEM (ii): JOINT COMPUTING & BANDWIDTH ALLOCATION -- INTERIOR POINT


In [10]:
def solve_resource_allocation(assignment: Dict[int, str],
                               tasks: List[Task],
                               tv_by_id: Dict[int, TaskVehicle],
                               rsu: RSU, service_vehicles: List[ServiceVehicle],
                               bw_total_mhz: float = None) -> Tuple[Dict[int, float], Dict[int, float]]:
    sv_by_id = {sv.id: sv for sv in service_vehicles}
    bmax = bw_total_mhz if bw_total_mhz is not None else CFG["BW_TOTAL_MHZ"]

    nodes_tasks: Dict[str, List[Task]] = {}
    for t in tasks:
        node = assignment[t.id]
        if node == NODE_LOCAL:
            continue
        nodes_tasks.setdefault(node, []).append(t)

    f_alloc: Dict[int, float] = {}

    for node, node_task_list in nodes_tasks.items():
        fmax = rsu.f_max_ghz if node == NODE_RSU else sv_by_id[node].f_max_ghz
        n = len(node_task_list)
        cycles = np.array([t.cycles_mc for t in node_task_list])

        if n == 1:
            f_opt = np.array([fmax])
        else:
            def obj(f):
                return np.sum(cycles / np.maximum(f, CFG["EPS"]))

            def grad(f):
                return -cycles / np.maximum(f, CFG["EPS"]) ** 2

            f0 = np.full(n, fmax / n)
            bounds = Bounds(lb=np.full(n, 1e-3), ub=np.full(n, fmax))
            lincon = LinearConstraint(np.ones((1, n)), lb=-np.inf, ub=fmax)
            res = minimize(obj, f0, jac=grad, method="trust-constr",
                            bounds=bounds, constraints=[lincon],
                            options=dict(maxiter=200, gtol=1e-8, xtol=1e-10))
            f_opt = res.x if res.success else fmax * np.sqrt(cycles) / np.sum(np.sqrt(cycles))

        for t, f in zip(node_task_list, f_opt):
            f_alloc[t.id] = float(f)

    b_alloc: Dict[int, float] = {}
    v2i_tasks = [t for t in tasks if assignment[t.id] == NODE_RSU]
    v2v_tasks = [t for t in tasks if assignment[t.id] not in (NODE_LOCAL, NODE_RSU)]

    for pool_tasks in (v2i_tasks, v2v_tasks):
        if not pool_tasks:
            continue
        n = len(pool_tasks)
        gains = []
        for t in pool_tasks:
            tv = tv_by_id[t.tv_id]
            node = assignment[t.id]
            exec_pos = rsu.pos if node == NODE_RSU else sv_by_id[node].pos
            gains.append(path_gain(abs(tv.pos - exec_pos)))
        gains = np.array(gains)

        if n == 1:
            b_opt = np.array([bmax])
        else:
            data = np.array([t.data_mb for t in pool_tasks])

            def obj(b):
                snr = CFG["TX_POWER_W"] * gains / CFG["NOISE_POWER_W"]
                rate = np.maximum(b, CFG["EPS"]) * np.log2(1.0 + snr)
                return np.sum(data / np.maximum(rate, CFG["EPS"]))

            b0 = np.full(n, bmax / n)
            bounds = Bounds(lb=np.full(n, 1e-3), ub=np.full(n, bmax))
            lincon = LinearConstraint(np.ones((1, n)), lb=-np.inf, ub=bmax)
            res = minimize(obj, b0, method="trust-constr",
                            bounds=bounds, constraints=[lincon],
                            options=dict(maxiter=200, gtol=1e-8, xtol=1e-10))
            b_opt = res.x if res.success else b0

        for t, b in zip(pool_tasks, b_opt):
            b_alloc[t.id] = float(b)

    return f_alloc, b_alloc

## 9. SUBPROBLEM (i): TASK OFFLOADING DECISION -- BRANCH & BOUND


In [11]:
class OffloadingBnB:
    def __init__(self, tasks, candidates, tv_by_id, rsu, service_vehicles,
                 models, bw_total_mhz: float = None, node_cap=CFG["BNB_NODE_CAP"]):
        self.tasks = tasks
        self.candidates = candidates
        self.tv_by_id = tv_by_id
        self.rsu = rsu
        self.service_vehicles = service_vehicles
        self.sv_by_id = {sv.id: sv for sv in service_vehicles}
        self.models = models
        self.bw_total_mhz = bw_total_mhz if bw_total_mhz is not None else CFG["BW_TOTAL_MHZ"]
        self.node_cap = node_cap

        self.best_cost = math.inf
        self.best_assign: Optional[Dict[int, str]] = None
        self.expanded_nodes = 0
        self.fell_back = False

        self.order = sorted(tasks, key=lambda t: t.deadline_s)

    def _optimistic_delay(self, task: Task, node_key, tau) -> float:
        tv = self.tv_by_id[task.tv_id]
        if node_key == NODE_LOCAL:
            f = tv.f_local_ghz
            t_comp = compute_delay_s(task.cycles_mc, f)
            return t_comp
        if node_key == NODE_RSU:
            f = self.rsu.f_max_ghz
            exec_pos = self.rsu.pos
        else:
            sv = self.sv_by_id[node_key]
            f = sv.f_max_ghz
            exec_pos = sv.pos
        dist = abs(tv.pos - exec_pos)
        rate = shannon_rate_mbps(self.bw_total_mhz, dist)
        t_off = trans_delay_s(task.data_mb, rate)
        t_comp = compute_delay_s(task.cycles_mc, f)
        return t_off + t_comp

    def _realistic_delay(self, task: Task, node_key, load_count: int) -> float:
        tv = self.tv_by_id[task.tv_id]
        if node_key == NODE_LOCAL:
            return compute_delay_s(task.cycles_mc, tv.f_local_ghz)
        if node_key == NODE_RSU:
            fmax, exec_pos = self.rsu.f_max_ghz, self.rsu.pos
        else:
            sv = self.sv_by_id[node_key]
            fmax, exec_pos = sv.f_max_ghz, sv.pos
        f_share = fmax / max(load_count, 1)
        dist = abs(tv.pos - exec_pos)
        rate = shannon_rate_mbps(self.bw_total_mhz / max(load_count, 1), dist)
        return trans_delay_s(task.data_mb, rate) + compute_delay_s(task.cycles_mc, f_share)

    def solve(self) -> Tuple[Dict[int, str], float]:
        assign: Dict[int, str] = {}
        node_load: Dict[str, int] = {}
        self._branch(0, assign, node_load, 0.0)
        if self.best_assign is None or self.fell_back:
            return self._greedy_fallback()
        return self.best_assign, self.best_cost

    def _branch(self, idx: int, assign: Dict[int, str], node_load: Dict[str, int], cost_so_far: float):
        self.expanded_nodes += 1
        if self.expanded_nodes > self.node_cap:
            self.fell_back = True
            return

        if idx == len(self.order):
            if cost_so_far < self.best_cost:
                self.best_cost = cost_so_far
                self.best_assign = dict(assign)
            return

        remaining_lb = 0.0
        for t in self.order[idx:]:
            cands = self.candidates[t.id]
            remaining_lb += min(self._optimistic_delay(t, n, tau) for n, tau in cands)
        if cost_so_far + remaining_lb >= self.best_cost:
            return

        task = self.order[idx]
        cands = self.candidates[task.id]
        cands_sorted = sorted(cands, key=lambda c: self._optimistic_delay(task, c[0], c[1]))

        for node_key, tau in cands_sorted:
            opt_delay = self._optimistic_delay(task, node_key, tau)
            if node_key != NODE_LOCAL and opt_delay > tau + task.deadline_s:
                continue
            if opt_delay > task.deadline_s * 3:
                continue

            new_load = dict(node_load)
            new_load[node_key] = new_load.get(node_key, 0) + 1
            step_cost = self._realistic_delay(task, node_key, new_load[node_key])

            assign[task.id] = node_key
            self._branch(idx + 1, assign, new_load, cost_so_far + step_cost)
            del assign[task.id]

            if self.expanded_nodes > self.node_cap:
                self.fell_back = True
                return

    def _greedy_fallback(self) -> Tuple[Dict[int, str], float]:
        assign: Dict[int, str] = {}
        node_load: Dict[str, int] = {}
        total = 0.0
        for task in self.order:
            cands = self.candidates[task.id]
            best_node, best_delay = None, math.inf
            for node_key, tau in cands:
                trial_load = node_load.get(node_key, 0) + 1
                d = self._realistic_delay(task, node_key, trial_load)
                if d < best_delay:
                    best_delay, best_node = d, node_key
            assign[task.id] = best_node
            node_load[best_node] = node_load.get(best_node, 0) + 1
            total += best_delay
        return assign, total

## 10. ALTERNATING OPTIMIZATION DRIVER


In [12]:
def run_alternating_optimization(tasks, tv_by_id, rsu, service_vehicles, models, bw_total_mhz: float = None):
    candidates = {t.id: get_candidates(t, tv_by_id[t.tv_id], rsu, service_vehicles) for t in tasks}
    bmax = bw_total_mhz if bw_total_mhz is not None else CFG["BW_TOTAL_MHZ"]

    assignment = None
    f_alloc, b_alloc = {}, {}
    fetch_info: Dict[int, Dict] = {}
    history = []

    prev_total = math.inf
    for it in range(CFG["AO_MAX_ITERS"]):
        bnb = OffloadingBnB(tasks, candidates, tv_by_id, rsu, service_vehicles, models, bw_total_mhz=bmax)
        assignment, _ = bnb.solve()

        f_alloc, b_alloc = solve_resource_allocation(assignment, tasks, tv_by_id, rsu, service_vehicles, bw_total_mhz=bmax)

        fetch_info = {}
        infeasible_tasks = []
        total_delay = 0.0
        for t in tasks:
            node = assignment[t.id]
            tv = tv_by_id[t.tv_id]
            f_here = f_alloc.get(t.id, tv.f_local_ghz if node == NODE_LOCAL else 1.0)
            b_off = b_alloc.get(t.id, bmax)
            b_fetch = bmax / 2.0

            has_model = node_has_model(node, t.model_id, rsu, service_vehicles, [tv])
            if node == NODE_LOCAL or has_model:
                t_off = 0.0 if node == NODE_LOCAL else trans_delay_s(t.data_mb, shannon_rate_mbps(b_off, 1.0))
                t_comp = compute_delay_s(t.cycles_mc, f_here)
                t_total = t_off + t_comp
                info = dict(mode="no_fetch_needed", t_off=t_off, t_fetch=0.0,
                            t_comp=t_comp, t_total=t_total,
                            feasible=(t_total <= t.deadline_s),
                            storage_ok=True, deadline_ok=(t_total <= t.deadline_s))
            else:
                info = lcsf_decide(t, node, tv, rsu, service_vehicles, models, f_here, b_off, b_fetch)
                if info["feasible"] and node not in (NODE_LOCAL, NODE_RSU):
                    sv = next(sv for sv in service_vehicles if sv.id == node)
                    used = sum(models[m].size_mb for m in sv.cached_models)
                    if used + models[t.model_id].size_mb <= sv.storage_mb:
                        sv.cached_models.add(t.model_id)

            fetch_info[t.id] = info
            total_delay += info["t_total"]
            if not info["feasible"]:
                infeasible_tasks.append(t.id)

        history.append(total_delay)

        changed = False
        for tid in infeasible_tasks:
            bad_node = assignment[tid]
            candidates[tid] = [c for c in candidates[tid] if c[0] != bad_node]
            if not candidates[tid]:
                candidates[tid] = [(NODE_LOCAL, 60.0)]
            changed = True

        if not changed and abs(prev_total - total_delay) < CFG["AO_TOL_S"]:
            break
        prev_total = total_delay

    return assignment, f_alloc, b_alloc, fetch_info, history

## 11. BASELINES (for comparison)


In [13]:
def baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, policy: str, bw_total_mhz: float = None) -> float:
    sv_by_id = {sv.id: sv for sv in service_vehicles}
    bmax = bw_total_mhz if bw_total_mhz is not None else CFG["BW_TOTAL_MHZ"]
    total = 0.0
    for t in tasks:
        tv = tv_by_id[t.tv_id]
        cands = get_candidates(t, tv, rsu, service_vehicles)

        if policy == "all_local":
            node = NODE_LOCAL
        elif policy == "all_rsu":
            node = NODE_RSU if any(c[0] == NODE_RSU for c in cands) else NODE_LOCAL
        elif policy == "random":
            node = random.choice(cands)[0]
        else:
            raise ValueError(policy)

        if node == NODE_LOCAL:
            total += compute_delay_s(t.cycles_mc, tv.f_local_ghz)
            continue

        exec_pos = rsu.pos if node == NODE_RSU else sv_by_id[node].pos
        f = rsu.f_max_ghz if node == NODE_RSU else sv_by_id[node].f_max_ghz
        dist = abs(tv.pos - exec_pos)
        rate = shannon_rate_mbps(bmax / 4, dist)
        t_off = trans_delay_s(t.data_mb, rate)

        has_model = node_has_model(node, t.model_id, rsu, service_vehicles, [tv])
        if has_model:
            t_fetch = 0.0
        else:
            rate_fetch = shannon_rate_mbps(bmax / 4, abs(rsu.pos - exec_pos))
            t_fetch = trans_delay_s(models[t.model_id].size_mb, rate_fetch)

        t_comp = compute_delay_s(t.cycles_mc, f / 2)
        total += t_off + t_fetch + t_comp
    return total

## 12. MAIN SIMULATION LOOP (multi time-slot, mobility, results & plots)


In [14]:
def generate_plots():
    print("Initializing structural data collection pipeline...")
    models = build_models()

    # ---------------- GRAPH 1: CONVERGENCE PROFILE ----------------
    print("Collecting convergence verification tracking trace (Graph 1)...")
    rsu, service_vehicles, task_vehicles = build_scenario(models, num_tv=12, num_sv=6)
    tasks = generate_tasks(task_vehicles, models, slot_idx=0)
    tv_by_id = {tv.id: tv for tv in task_vehicles}
    _, _, _, _, convergence_history = run_alternating_optimization(tasks, tv_by_id, rsu, service_vehicles, models)
    
    # Fill remaining iterative log indices to guarantee standard execution tracking size
    while len(convergence_history) < CFG["AO_MAX_ITERS"]:
        convergence_history.append(convergence_history[-1])

    plt.figure(figsize=(6, 4.5))
    plt.plot(range(1, CFG["AO_MAX_ITERS"] + 1), convergence_history, 'ro-', linewidth=1.5, label=r'$N=18, B_u=20$ MHz')
    plt.xlabel('Number of iterations')
    plt.ylabel('Total task completion delay (s)')
    plt.title('The convergence property of the JOSTR algorithm')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig1_convergence.png', dpi=200)
    plt.close()

    # ---------------- GRAPH 2: DELAY VS VEHICLE DENSITY (N) ----------------
    print("Collecting vehicle density parametric metrics sweep (Graph 2)...")
    # Mapping N to distinct proportional clusters: Total Vehicles N = Task + Service
    N_steps = [6, 12, 18, 24, 30]
    ao_delays, local_delays, rsu_delays, rand_delays = [], [], [], []

    for n_total in N_steps:
        # Balanced assignment ratio matching environment profiles
        n_sv = max(2, int(n_total * 0.33))
        n_tv = n_total - n_sv
        
        rsu, service_vehicles, task_vehicles = build_scenario(models, num_tv=n_tv, num_sv=n_sv)
        tasks = generate_tasks(task_vehicles, models, slot_idx=1)
        tv_by_id = {tv.id: tv for tv in task_vehicles}

        _, _, _, _, hist = run_alternating_optimization(tasks, tv_by_id, rsu, service_vehicles, models)
        ao_delays.append(hist[-1])
        local_delays.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "all_local"))
        rsu_delays.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "all_rsu"))
        rand_delays.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "random"))

    plt.figure(figsize=(6, 4.5))
    plt.plot(N_steps, ao_delays, 'r-o', label='Proposed (JOSTR)')
    plt.plot(N_steps, local_delays, 'b-x', label='All-Local')
    plt.plot(N_steps, rsu_delays, 'g-^', label='All-RSU')
    plt.plot(N_steps, rand_delays, 'm-s', label='Random')
    plt.xlabel('Number of vehicles N')
    plt.ylabel('Total task completion delay (s)')
    plt.title('Total task completion delay under different numbers of vehicles N')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig2_delay_vs_N.png', dpi=200)
    plt.close()

    # ---------------- GRAPH 3: DELAY VS UPLINK BANDWIDTH (Bu)[cite: 4] ----------------
    print("Collecting uplink channel optimization metrics sweep (Graph 3)[cite: 4]...")
    Bu_steps = [10.0, 15.0, 20.0, 25.0, 30.0]  # Bandwidth variance array (MHz)[cite: 4]
    ao_delays_bw, local_delays_bw, rsu_delays_bw, rand_delays_bw = [], [], [], []

    # Fix layout sizing across test profiles
    rsu, service_vehicles, task_vehicles = build_scenario(models, num_tv=12, num_sv=6)
    tasks = generate_tasks(task_vehicles, models, slot_idx=2)
    tv_by_id = {tv.id: tv for tv in task_vehicles}

    for b_pool in Bu_steps:
        _, _, _, _, hist = run_alternating_optimization(tasks, tv_by_id, rsu, service_vehicles, models, bw_total_mhz=b_pool)
        ao_delays_bw.append(hist[-1])
        local_delays_bw.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "all_local", bw_total_mhz=b_pool))
        rsu_delays_bw.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "all_rsu", bw_total_mhz=b_pool))
        rand_delays_bw.append(baseline_delay(tasks, tv_by_id, rsu, service_vehicles, models, "random", bw_total_mhz=b_pool))

    plt.figure(figsize=(6, 4.5))
    plt.plot(Bu_steps, ao_delays_bw, 'r-o', label='Proposed (JOSTR)')
    plt.plot(Bu_steps, local_delays_bw, 'b-x', label='All-Local')
    plt.plot(Bu_steps, rsu_delays_bw, 'g-^', label='All-RSU')
    plt.plot(Bu_steps, rand_delays_bw, 'm-s', label='Random')
    plt.xlabel('Uplink Bandwidth Bu (MHz)')
    plt.ylabel('Total task completion delay (s)')
    plt.title('Total task completion delay under different uplink bandwidths Bu')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig('fig3_delay_vs_Bu.png', dpi=200)
    plt.close()

    print("Data structures processing complete. Validation figures saved safely inside local working environment directory.")


if __name__ == "__main__":
    generate_plots()

Initializing structural data collection pipeline...


c:\VEC\.venv\Lib\site-packages\scipy\optimize\_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
c:\VEC\.venv\Lib\site-packages\scipy\optimize\_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
c:\VEC\.venv\Lib\site-packages\scipy\optimize\_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
 

Data structures processing complete. Validation figures saved safely inside local working environment directory.
